# 04 — Sémantique : un graphe de connaissances des vols, interrogeable en SPARQL

```
ontologie/vols.ttl (schéma) ─┐
sem_aeroports ───────────────┤
sem_compagnies ──────────────┼─► rdflib ─► ontologie/graphe_vols.ttl ─► Fuseki :3030 (SPARQL)
sem_routes (top 50) ─────────┤                                        └─► repli : rdflib
sem_vols (500 vols) ─────────┘
```

Les tables gold répondent aux questions **prévues** (une question = une table). Le graphe sert aux questions
qui **traversent** plusieurs notions à la fois : *« au départ de quels hubs les retards sont-ils surtout
contrôlables ? »* relie un vol à son aéroport, l'aéroport à sa classe (hub ou non), le retard à ses causes et
chaque cause à sa catégorie. L'ontologie décrit ces notions une seule fois ; les requêtes s'appuient sur les
classes au lieu de listes codées en dur.

**Contenu du graphe**

| Élément | Source | Nombre |
|---|---|---|
| Schéma : classes, propriétés, les 5 causes | `ontologie/vols.ttl` | — |
| Tous les aéroports desservis, avec leur État | `sem_aeroports` | 348 |
| Toutes les compagnies | `sem_compagnies` | 15 |
| Les routes au plus fort `score_priorite` | `sem_routes` | 50 |
| Un échantillon de vols sur ces routes, avec leurs retards par cause | `sem_vols` | 500 |
| Liens `owl:sameAs` des hubs vers Wikidata | table ci-dessous | 30 |

Les tables `sem_*` sont préparées par `03_gold` (section 9) à partir du silver et copiées dans PostgreSQL : ce
notebook n'a besoin que de pandas et de rdflib, **sans session Spark**. Le graphe ne contient qu'un échantillon
de vols : il sert à **montrer** ce qu'apporte une ontologie, pas à remplacer les tables gold pour les chiffres
nationaux.

## 1. Bibliothèques et lecture des tables `sem_*`

`rdflib` (graphes RDF en Python) n'est pas dans l'image Jupyter : la cellule l'installe au premier lancement.
`requests` sert à parler à Fuseki (section 6).

In [ ]:
import subprocess
import sys
from decimal import Decimal
from pathlib import Path

sys.path.append("/home/jovyan/work")
from commun.donnees import lire

try:
    import rdflib
except ImportError:   # absent de l'image quay.io/jupyter/pyspark-notebook : installé une seule fois
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdflib"], check=True)
    import rdflib

import pandas as pd
import requests
from rdflib import Graph, Literal, Namespace
from rdflib.namespace import OWL, RDF, RDFS, XSD

# Racine du dépôt : /home/jovyan/work dans le conteneur Jupyter, le dossier parent sinon
RACINE = Path("/home/jovyan/work") if Path("/home/jovyan/work").exists() else Path.cwd().parent
SCHEMA = RACINE / "ontologie" / "vols.ttl"
SORTIE = RACINE / "ontologie" / "graphe_vols.ttl"
print(f"rdflib {rdflib.__version__} — schéma lu dans {SCHEMA.relative_to(RACINE).as_posix()}")

def milliers(n):
    """1234567 → « 1 234 567 » (séparateur de milliers français)."""
    return f"{int(n):,}".replace(",", " ")

In [ ]:
aeroports = lire("sem_aeroports")
compagnies = lire("sem_compagnies")
routes = lire("sem_routes")
vols = lire("sem_vols")

for nom, table in [("sem_aeroports", aeroports), ("sem_compagnies", compagnies),
                   ("sem_routes", routes), ("sem_vols", vols)]:
    print(f"{nom:<15} {len(table):>4} lignes   colonnes : {', '.join(table.columns)}")

## 2. Le schéma (T-Box) : `ontologie/vols.ttl`

On charge d'abord l'ontologie dans un graphe vide : les instances viendront s'y ajouter. Le schéma est
lui-même fait de triplets, on peut donc l'interroger en SPARQL comme les données. Ci-dessous, les classes et
leur classe mère : on retrouve les deux hiérarchies du modèle, **Hub ⊂ Aeroport** et
**CauseControlable / CauseNonControlable ⊂ CauseRetard**.

Espaces de noms : `vols:` pour le vocabulaire ; un espace par type d'instance pour les données (`aer:ATL`,
`cie:DL`, `route:LAX-SFO`, `vol:20240101_B6_623_JFK_14`, `etat:GA`, `imp:` pour les imputations), et `wd:`
pour Wikidata.

In [ ]:
VOLS = Namespace("http://example.org/vols/ontologie#")
AER = Namespace("http://example.org/vols/aeroport/")
CIE = Namespace("http://example.org/vols/compagnie/")
ROUTE = Namespace("http://example.org/vols/route/")
VOL = Namespace("http://example.org/vols/vol/")
ETAT = Namespace("http://example.org/vols/etat/")
IMP = Namespace("http://example.org/vols/imputation/")
WD = Namespace("http://www.wikidata.org/entity/")
PREFIXES = {"vols": VOLS, "aer": AER, "cie": CIE, "route": ROUTE, "vol": VOL, "etat": ETAT, "imp": IMP,
            "wd": WD, "owl": OWL, "rdfs": RDFS, "xsd": XSD}

g = Graph()
g.parse(SCHEMA, format="turtle")
for prefixe, espace in PREFIXES.items():
    g.bind(prefixe, espace)
n_schema = len(g)
print(f"Schéma : {n_schema} triplets")

hierarchie = g.query("""
    SELECT ?classe ?libelle ?mere WHERE {
        ?classe a owl:Class ; rdfs:label ?libelle .
        FILTER (isIRI(?classe) && lang(?libelle) = "fr")
        OPTIONAL { ?classe rdfs:subClassOf ?mere }
    }
    ORDER BY ?mere ?classe
""", initNs=PREFIXES)
pd.DataFrame([(g.qname(c), str(l), g.qname(m) if m else "") for c, l, m in hierarchie],
             columns=["classe", "libellé", "sous-classe de"])

## 3. Les instances (A-Box)

**Conventions**
- Chaque valeur est écrite avec son type XSD (`xsd:decimal`, `xsd:integer`, `xsd:date`, `xsd:boolean`) : SPARQL
  peut alors comparer, sommer et trier sans conversion.
- **Une valeur manquante ne donne aucun triplet.** RDF n'a pas de `NULL` : un vol annulé n'a simplement pas de
  `vols:retardArriveeMin`. Les requêtes le traitent avec `OPTIONAL` ou `FILTER NOT EXISTS`.
- Les fonctions ci-dessous centralisent ces règles : `ajouter()` n'écrit le triplet que si la valeur existe.

In [ ]:
def vide(valeur):
    """Vrai pour une valeur manquante (None, NaN, NaT) : on n'écrit alors aucun triplet."""
    return valeur is None or (not isinstance(valeur, str) and pd.isna(valeur))


def decimal(valeur, chiffres=4):
    """Littéral xsd:decimal arrondi (format fixe, jamais de notation 1e-05)."""
    return Literal(format(Decimal(str(round(float(valeur), chiffres))), "f"), datatype=XSD.decimal)


def entier(valeur):
    return Literal(int(valeur))                      # xsd:integer


def texte(valeur):
    """Chaîne ; un nombre entier lu en flottant (623.0) redevient « 623 »."""
    if isinstance(valeur, float) and valeur.is_integer():
        valeur = int(valeur)
    return Literal(str(valeur))


def ajouter(sujet, propriete, valeur, convertir=texte):
    if not vide(valeur):
        g.add((sujet, propriete, convertir(valeur)))